# Eksperimen Word Embedding Skip-Gram & Klasifikasi Naive Bayes
### Analisis Komparatif 5 Kasus Hyperparameter Skip-Gram pada Klasifikasi Teks Berita (Detik Sport vs Detik Finance)

Notebook ini menyajikan implementasi lengkap penambangan teks (*Text Mining*) menggunakan arsitektur **Word2Vec Skip-Gram** sebagai ekstraktor fitur representasi kata/dokumen (*Word & Document Embeddings*), yang kemudian diklasifikasikan menggunakan algoritma **Gaussian Naive Bayes**.

---

### Tujuan Eksperimen:
1. Mengonstruksi **5 Kasus (*Cases*) Hyperparameter Skip-Gram** yang bervariasi (`vector_size`, `window`, `min_count`, `epochs`, `negative sampling`).
2. Mentransformasikan dokumen teks berita menjadi representasi vektor padat (*dense document vectors*) melalui teknik **Average Word Embedding (Mean Pooling)**.
3. Melatih dan mengevaluasi model klasifikasi **Gaussian Naive Bayes** pada masing-masing representasi fitur dari ke-5 kasus tersebut.
4. Mengukur performa menggunakan metrik **Akurasi Data Uji (Test Accuracy)**, **Precision**, **Recall**, **F1-Score**, serta **Stratified 5-Fold Cross Validation**.
5. Menyajikan perbandingan komparatif, visualisasi grafik, analisis kemiripan kata semantik (*word similarity*), dan fungsi inferensi interaktif untuk klasifikasi berita baru.


## 1. Setup Lingkungan & Import Library

Pada tahap ini, kita memuat seluruh pustaka yang diperlukan:
- **Gensim (`Word2Vec`)**: Pustaka utama untuk melatih model representasi kata Skip-Gram (`sg=1`).
- **Scikit-Learn**: Modul untuk pemodelan machine learning (`GaussianNB`), pembagian data (`train_test_split`), validasi silang (`StratifiedKFold`, `cross_val_score`), serta metrik evaluasi (`accuracy_score`, `classification_report`, `confusion_matrix`).
- **Pandas & NumPy**: Manipulasi struktur data tabular dan operasi matriks numerik.
- **Matplotlib & Seaborn**: Visualisasi grafik perbandingan dan heatmap matriks konfusi.


In [ ]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Library NLP & Word Embedding
from gensim.models import Word2Vec

# Library Machine Learning & Evaluasi
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Konfigurasi Tampilan Grafik & Warning
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120

# Seed untuk reproduktifitas
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Semua pustaka berhasil dimuat dengan sukses!")


## 2. Pemuatan & Eksplorasi Dataset Berita

Dataset yang digunakan merupakan kumpulan 200 artikel berita Detikcom (100 berita **Sport** dan 100 berita **Finance**) yang telah melalui tahap preprocessing teks (pembersihan noise regex, case folding, stopword removal, dan lemmatization/stemming Sastrawi).

Berkas utama yang dimuat adalah `data_berita_processed.csv`.


In [ ]:
# Lokasi file dataset
DATASET_PATH = 'data_berita_processed.csv'

if os.path.exists(DATASET_PATH):
    df = pd.read_csv(DATASET_PATH)
    print(f"Dataset berhasil dimuat dari '{DATASET_PATH}'!")
else:
    raise FileNotFoundError(f"File '{DATASET_PATH}' tidak ditemukan. Pastikan file berada di direktori yang sama.")

# Ringkasan informasi dataset
print(f"Bentuk Dataset        : {df.shape[0]} baris x {df.shape[1]} kolom")
print(f"Distribusi Kategori   :\n{df['kategori'].value_counts()}\n")

# Menampilkan 5 sampel data pertama
display_cols = [c for c in ['judul', 'kategori', 'isi_berita'] if c in df.columns]
df[display_cols].head()


## 3. Tokenisasi Korpus & Fungsi Representasi Dokumen

Model Word2Vec Skip-Gram membutuhkan input berupa daftar token kata (*list of lists of tokens*).
Setelah model Skip-Gram terlatih dan menghasilkan vektor kata $\mathbf{v}_w \in \mathbb{R}^d$, dokumen berita direpresentasikan menjadi satu vektor tunggal menggunakan metode **Average Word Embedding (Mean Pooling)**:

$$\mathbf{v}_{D} = \frac{1}{|D_{\text{in-vocab}}|} \sum_{w \in D_{\text{in-vocab}}} \mathbf{v}_w$$

Di mana:
- $D_{\text{in-vocab}}$ adalah kumpulan kata dalam dokumen yang terdaftar pada kosakata (*vocabulary*) model Skip-Gram.
- Jika tidak ada kata yang terdaftar dalam kosakata, dokumen direpresentasikan sebagai vektor nol $\mathbf{0} \in \mathbb{R}^d$.


In [ ]:
# 1. Tokenisasi teks berita menjadi daftar token kata
tokenized_corpus = [str(text).split() for text in df['isi_berita']]

# 2. Encoding Label Target: Sport = 1, Finance = 0
y = df['kategori'].map({'sport': 1, 'finance': 0}).values

# 3. Fungsi Ekstraksi Rata-Rata Vektor Kata (Mean Word Embedding) per Dokumen
def get_document_vector(tokens, model, vector_size):
    '''
    Menghitung representasi vektor dokumen dengan merata-ratakan vektor kata penyusunnya.
    '''
    # Mengambil vektor untuk kata-kata yang ada dalam kosakata model
    valid_vectors = [model.wv[word] for word in tokens if word in model.wv]
    
    if len(valid_vectors) > 0:
        return np.mean(valid_vectors, axis=0)
    else:
        # Fallback vektor nol jika tidak ada kata yang cocok
        return np.zeros(vector_size)

print(f"Total Dokumen Dikomputasi : {len(tokenized_corpus)}")
print(f"Contoh Token Dokumen 1    : {tokenized_corpus[0][:12]} ...")
print(f"Distribusi Label (y)      : {np.bincount(y)} (0: Finance, 1: Sport)")


## 4. Perancangan 5 Kasus (*Cases*) Hyperparameter Skip-Gram

Arsitektur **Skip-Gram (`sg=1`)** memprediksi kata-kata konteks di sekitarnya berdasarkan kata target sentral. Untuk menguji sensitivitas model terhadap representasi semantik dan performa klasifikasi Naive Bayes, dirancang **5 Kasus Hyperparameter**:

| Kasus | Dimensi Vektor (`vector_size`) | Jendela Konteks (`window`) | Frekuensi Minimal (`min_count`) | Iterasi Pelatihan (`epochs`) | Negative Sampling (`negative`) | Rasionalisasi & Karakteristik |
| :---: | :---: | :---: | :---: | :---: | :---: | :--- |
| **Case 1** | **100** | **5** | **2** | **20** | **5** | **Baseline Standar**: Konfigurasi umum yang menyeimbangkan dimensi dan jendela konteks. |
| **Case 2** | **50** | **2** | **1** | **20** | **5** | **Dimensi Ringan & Jendela Sempit**: Fokus pada hubungan sintaksis lokal terdekat tanpa membuang kata langka. |
| **Case 3** | **200** | **10** | **2** | **30** | **5** | **Dimensi Tinggi & Jendela Luas**: Menangkap konteks semantik makro dan tema luas lintas paragraf. |
| **Case 4** | **100** | **5** | **3** | **50** | **5** | **Filter Kosakata Selektif & Konvergensi Panjang**: Memangkas kata bising (*noise*) dan iterasi lebih matang. |
| **Case 5** | **150** | **7** | **2** | **30** | **10** | **Negative Sampling Intensif**: Melipatgandakan sampel kontras negatif untuk diskriminasi fitur lebih tajam. |


In [ ]:
# Definisi 5 Kasus Hyperparameter Skip-Gram
cases_config = [
    {
        'case_id': 1,
        'name': 'Case 1 (Baseline: Dim 100, Win 5, MinCount 2, Epoch 20, Neg 5)',
        'vector_size': 100,
        'window': 5,
        'min_count': 2,
        'epochs': 20,
        'negative': 5,
        'description': 'Konfigurasi Baseline Standar'
    },
    {
        'case_id': 2,
        'name': 'Case 2 (Dim 50, Win 2, MinCount 1, Epoch 20, Neg 5)',
        'vector_size': 50,
        'window': 2,
        'min_count': 1,
        'epochs': 20,
        'negative': 5,
        'description': 'Dimensi Ringan & Jendela Sintaksis Sempit'
    },
    {
        'case_id': 3,
        'name': 'Case 3 (Dim 200, Win 10, MinCount 2, Epoch 30, Neg 5)',
        'vector_size': 200,
        'window': 10,
        'min_count': 2,
        'epochs': 30,
        'negative': 5,
        'description': 'Dimensi Tinggi & Jendela Topikal Luas'
    },
    {
        'case_id': 4,
        'name': 'Case 4 (Dim 100, Win 5, MinCount 3, Epoch 50, Neg 5)',
        'vector_size': 100,
        'window': 5,
        'min_count': 3,
        'epochs': 50,
        'negative': 5,
        'description': 'Filter Kosakata Selektif & Konvergensi Mendalam'
    },
    {
        'case_id': 5,
        'name': 'Case 5 (Dim 150, Win 7, MinCount 2, Epoch 30, Neg 10)',
        'vector_size': 150,
        'window': 7,
        'min_count': 2,
        'epochs': 30,
        'negative': 10,
        'description': 'Negative Sampling Intensif & Dimensi Moderat'
    }
]

df_cases = pd.DataFrame(cases_config)[['case_id', 'vector_size', 'window', 'min_count', 'epochs', 'negative', 'description']]
df_cases


## 5. Pelatihan Model Skip-Gram & Ekstraksi Fitur Dokumen

Setiap konfigurasi dilatih secara mandiri menggunakan modul `gensim.models.Word2Vec` dengan parameter `sg=1` (Skip-Gram). Setelah pelatihan selesai, seluruh dokumen dalam korpus dikonversi menjadi matriks fitur dokumen $\mathbf{X} \in \mathbb{R}^{N \times d}$ menggunakan fungsi perataan rata-rata (*Mean Word Embedding*).


In [ ]:
# Wadah penyimpanan model dan matriks fitur dokumen
trained_sg_models = {}
feature_matrices = {}

print("=== MEMULAI PELATIHAN 5 MODEL SKIP-GRAM ===")

for cfg in cases_config:
    cid = cfg['case_id']
    name = cfg['name']
    
    # 1. Melatih Model Word2Vec Skip-Gram (sg=1)
    sg_model = Word2Vec(
        sentences=tokenized_corpus,
        sg=1,                        # Arsitektur Skip-Gram
        vector_size=cfg['vector_size'],
        window=cfg['window'],
        min_count=cfg['min_count'],
        epochs=cfg['epochs'],
        negative=cfg['negative'],
        seed=RANDOM_STATE,
        workers=4
    )
    
    # Simpan model
    trained_sg_models[cid] = sg_model
    vocab_size = len(sg_model.wv)
    
    # 2. Ekstraksi Matriks Fitur Dokumen
    X_mat = np.array([
        get_document_vector(tokens, sg_model, cfg['vector_size'])
        for tokens in tokenized_corpus
    ])
    feature_matrices[cid] = X_mat
    
    print(f"[{cid}/5] {name}")
    print(f"      -> Ukuran Kosakata (Vocab Size): {vocab_size} kata")
    print(f"      -> Bentuk Matriks Fitur (X)    : {X_mat.shape} (Dokumen x Dimensi)\n")

print("Seluruh 5 model Skip-Gram berhasil dilatih dan diekstraksi ke matriks fitur!")


## 6. Eksplorasi Semantik Vektor Kata (*Word Similarity Analysis*)

Untuk menguji secara kualitatif bagaimana masing-masing hyperparameter memengaruhi pemahaman semantik model, kita menganalisis kata-kata yang paling mirip (*cosine similarity tertinggi*) dengan kata kunci:
- **Kategori Sport**: kata `'atlet'` atau `'tanding'`
- **Kategori Finance**: kata `'saham'` atau `'investasi'`


In [ ]:
def eksplorasi_kemiripan_kata(target_word, top_n=5):
    print(f"\n--- 5 Kata Paling Mirip dengan '{target_word}' Lintas 5 Kasus Skip-Gram ---")
    data_sim = {}
    
    for cfg in cases_config:
        cid = cfg['case_id']
        model = trained_sg_models[cid]
        case_col = f"Case {cid}"
        
        if target_word in model.wv:
            similars = model.wv.most_similar(target_word, topn=top_n)
            data_sim[case_col] = [f"{word} ({score:.3f})" for word, score in similars]
        else:
            data_sim[case_col] = ["(Kata tidak ada di vocab)"] * top_n
            
    df_sim = pd.DataFrame(data_sim, index=[f"Rank {i+1}" for i in range(top_n)])
    return df_sim

# Uji Coba Eksplorasi Kata
display(eksplorasi_kemiripan_kata('saham'))
display(eksplorasi_kemiripan_kata('atlet'))


## 7. Pemodelan Klasifikasi Naive Bayes (Gaussian Naive Bayes)

### Mengapa Menggunakan Gaussian Naive Bayes?
Berbeda dengan representasi frekuensi diskrit seperti Bag-of-Words atau TF-IDF yang bernilai non-negatif (cocok untuk `MultinomialNB`), **Word Embedding Skip-Gram** memetakan kata ke dalam ruang vektor kontinu $\mathbb{R}^d$ yang memiliki nilai riil negatif maupun positif:

$$\mathbf{x} \in \mathbb{R}^d, \quad x_i \in (-\infty, \infty)$$

Oleh karena itu, varian Naive Bayes yang valid secara matematis adalah **Gaussian Naive Bayes (`GaussianNB`)**, yang memodelkan probabilitas fitur kontinu berdasarkan distribusi normal (*Gaussian Normal Distribution*):

$$P(x_i \mid y) = \frac{1}{\sqrt{2\pi\sigma_{y,i}^2}} \exp\left(-\frac{(x_i - \mu_{y,i})^2}{2\sigma_{y,i}^2}\right)$$

### Protokol Pengujian:
1. **Stratified Train-Test Split (80:20)**: 160 sampel latih dan 40 sampel uji dengan proporsi kelas seimbang.
2. **Stratified 5-Fold Cross Validation**: Menguji stabilitas model di seluruh lipatan korpus berita.
3. **Metrik Evaluasi**: Accuracy, Precision, Recall, F1-Score, dan Confusion Matrix.


In [ ]:
# Protokol Pembagian Data dan Validasi Silang
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

evaluation_results = []
trained_nb_models = {}
predictions_test = {}

print("=== MEMULAI PELATIHAN & EVALUASI GAUSSIAN NAIVE BAYES ===")

for cfg in cases_config:
    cid = cfg['case_id']
    name = cfg['name']
    X_current = feature_matrices[cid]
    
    # 1. Pembagian Data Latih (80%) dan Data Uji (20%)
    X_train, X_test, y_train, y_test = train_test_split(
        X_current, y,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=y
    )
    
    # 2. Inisialisasi & Pelatihan Model Gaussian Naive Bayes
    nb_classifier = GaussianNB()
    nb_classifier.fit(X_train, y_train)
    trained_nb_models[cid] = nb_classifier
    
    # 3. Prediksi pada Data Uji (40 dokumen)
    y_pred = nb_classifier.predict(X_test)
    predictions_test[cid] = y_pred
    
    # 4. Perhitungan Metrik Evaluasi pada Data Uji
    acc = accuracy_score(y_test, y_pred) * 100
    prec = precision_score(y_test, y_pred, average='weighted') * 100
    rec = recall_score(y_test, y_pred, average='weighted') * 100
    f1 = f1_score(y_test, y_pred, average='weighted') * 100
    
    # 5. Stratified 5-Fold Cross Validation pada Seluruh Data
    cv_scores = cross_val_score(GaussianNB(), X_current, y, cv=cv_strategy, scoring='accuracy') * 100
    cv_mean = cv_scores.mean()
    cv_std = cv_scores.std()
    
    # Simpan Hasil Evaluasi
    res_entry = {
        'Case': f"Case {cid}",
        'Deskripsi': cfg['description'],
        'Dimensi': cfg['vector_size'],
        'Window': cfg['window'],
        'Min Count': cfg['min_count'],
        'Epochs': cfg['epochs'],
        'Negative': cfg['negative'],
        'Test Acc (%)': acc,
        '5-Fold CV Mean (%)': cv_mean,
        '5-Fold CV Std (%)': cv_std,
        'Precision (%)': prec,
        'Recall (%)': rec,
        'F1-Score (%)': f1
    }
    evaluation_results.append(res_entry)
    
    print(f"[{cid}/5] {name}")
    print(f"      -> Akurasi Data Uji (Test Acc) : {acc:.2f}%")
    print(f"      -> 5-Fold Cross Validation     : {cv_mean:.2f}% (+/- {cv_std:.2f}%)\n")

print("Evaluasi Naive Bayes pada seluruh 5 Kasus telah selesai!")


## 8. Tabel Ringkasan Komparasi Performa 5 Kasus

Berikut adalah tabel rekapitulasi performa klasifikasi Gaussian Naive Bayes pada 5 variasi representasi fitur Skip-Gram:


In [ ]:
# Membuat DataFrame Rekapitulasi
df_summary = pd.DataFrame(evaluation_results)

# Menampilkan tabel komparasi utama
styled_summary = df_summary[['Case', 'Deskripsi', 'Dimensi', 'Window', 'Epochs', 'Test Acc (%)', '5-Fold CV Mean (%)', 'F1-Score (%)']]
display(styled_summary)

# Menentukan Kasus Terbaik Berdasarkan 5-Fold CV Mean
best_case_row = df_summary.loc[df_summary['5-Fold CV Mean (%)'].idxmax()]
best_case_id = int(best_case_row['Case'].replace('Case ', ''))

print(f"\n🏆 KASUS TERBAIK: {best_case_row['Case']} ({best_case_row['Deskripsi']})")
print(f"   Akurasi Data Uji : {best_case_row['Test Acc (%)']:.2f}%")
print(f"   Skor 5-Fold CV   : {best_case_row['5-Fold CV Mean (%)']:.2f}% (+/- {best_case_row['5-Fold CV Std (%)']:.2f}%)")


## 9. Visualisasi Grafik Perbandingan & Confusion Matrix

Dua visualisasi analitis disajikan untuk memperjelas komparasi performa:
1. **Diagram Batang Komparatif (Grouped Bar Chart)**: Membandingkan Test Accuracy dan 5-Fold CV Mean di seluruh ke-5 kasus.
2. **Confusion Matrix Heatmap**: Mengamati distribusi prediksi benar vs salah untuk kasus dengan skor validasi silang tertinggi.


In [ ]:
# 1. Visualisasi Diagram Batang Komparasi Akurasi 5 Kasus
fig, ax = plt.subplots(figsize=(12, 6))

cases_labels = [f"Case {r['case_id']}\n(Dim {r['vector_size']}, Win {r['window']})" for r in cases_config]
test_accuracies = df_summary['Test Acc (%)'].values
cv_means = df_summary['5-Fold CV Mean (%)'].values
cv_stds = df_summary['5-Fold CV Std (%)'].values

x_pos = np.arange(len(cases_labels))
bar_width = 0.35

rects1 = ax.bar(x_pos - bar_width/2, test_accuracies, bar_width, label='Akurasi Data Uji (Test Acc)', color='#2563eb', alpha=0.9)
rects2 = ax.bar(x_pos + bar_width/2, cv_means, bar_width, yerr=cv_stds, capsize=5, label='5-Fold CV Mean (+/- Std)', color='#10b981', alpha=0.9)

ax.set_ylabel('Persentase Performa (%)', fontsize=12, fontweight='bold')
ax.set_title('Komparasi Akurasi Gaussian Naive Bayes pada 5 Kasus Hyperparameter Skip-Gram', fontsize=13, fontweight='bold', pad=15)
ax.set_xticks(x_pos)
ax.set_xticklabels(cases_labels, fontsize=10, fontweight='bold')
ax.set_ylim(70, 105)
ax.legend(loc='lower right', fontsize=11, frameon=True)
ax.grid(axis='y', linestyle='--', alpha=0.6)

# Label angka di atas batang grafik
def beri_label_batang(rects):
    for rect in rects:
        h = rect.get_height()
        ax.annotate(f'{h:.1f}%',
                    xy=(rect.get_x() + rect.get_width() / 2, h),
                    xytext=(0, 4), textcoords="offset points",
                    ha='center', va='bottom', fontsize=9, fontweight='bold')

beri_label_batang(rects1)
beri_label_batang(rects2)

plt.tight_layout()
plt.show()

# 2. Visualisasi Heatmap Confusion Matrix untuk Model Terbaik
best_pred = predictions_test[best_case_id]
cm = confusion_matrix(y_test, best_pred)
labels = ['Finance (0)', 'Sport (1)']

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, cbar=False, annot_kws={'size': 14, 'weight': 'bold'})
plt.title(f'Confusion Matrix: Case {best_case_id} (Model Terbaik)', fontsize=12, fontweight='bold', pad=12)
plt.xlabel('Prediksi Model Naive Bayes', fontsize=11, fontweight='bold')
plt.ylabel('Aktual (Ground Truth)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

# Laporan Klasifikasi Rinci untuk Model Terbaik
print(f"=== Laporan Klasifikasi Rinci: Case {best_case_id} ===")
print(classification_report(y_test, best_pred, target_names=labels))


## 10. Fungsi Prediksi Teks Berita Baru (*Interactive Inference*)

Fungsi berikut disiapkan secara mandiri untuk menguji teks berita baru dari pengguna. Alur kerjanya meliputi:
1. Pembersihan noise dasar & tokenisasi teks input.
2. Ekstraksi vektor dokumen rata-rata (*Mean Word Embedding*) menggunakan model Skip-Gram terpilih.
3. Klasifikasi menggunakan model Gaussian Naive Bayes terlatih.
4. Menghasilkan label kategori (`Sport` atau `Finance`) beserta probabilitas keyakinan (*Confidence Score*).


In [ ]:
def prediksi_kategori_berita(teks_input, case_terpilih=best_case_id):
    '''
    Memprediksi kategori berita (Sport vs Finance) untuk teks baru.
    '''
    # 1. Ambil model Skip-Gram dan Naive Bayes yang sesuai
    sg_model = trained_sg_models[case_terpilih]
    nb_model = trained_nb_models[case_terpilih]
    vector_size = cases_config[case_terpilih - 1]['vector_size']
    
    # 2. Pembersihan sederhana & tokenisasi
    clean_text = re.sub(r'[^a-zA-Z\s]', ' ', teks_input.lower())
    tokens = clean_text.split()
    
    # 3. Transformasi ke Mean Word Embedding
    doc_vec = get_document_vector(tokens, sg_model, vector_size).reshape(1, -1)
    
    # 4. Prediksi Kategori & Probabilitas
    pred_class = nb_model.predict(doc_vec)[0]
    pred_prob = nb_model.predict_proba(doc_vec)[0]
    
    nama_kategori = 'Sport' if pred_class == 1 else 'Finance'
    confidence = pred_prob[pred_class] * 100
    
    print("=" * 60)
    print(f"Teks Berita       : {teks_input[:100]} ...")
    print(f"Model Digunakan   : Case {case_terpilih} ({cases_config[case_terpilih - 1]['description']})")
    print(f"Hasil Prediksi    : >> {nama_kategori.upper()} <<")
    print(f"Tingkat Keyakinan : {confidence:.2f}% (Finance: {pred_prob[0]*100:.1f}%, Sport: {pred_prob[1]*100:.1f}%)")
    print("=" * 60)
    
    return nama_kategori, confidence

# --- Uji Coba Sampel Berita Baru ---
print("UJI COBA PREDIKSI BERITA INTERAKTIF:\n")

# Sampel 1: Kalimat bernuansa Olahraga
sampel_sport = "Pebulutangkis tunggal putra Indonesia berhasil melaju ke final turnamen BWF World Tour setelah menumbangkan unggulan pertama dalam pertandingan rubber game yang menegangkan."
prediksi_kategori_berita(sampel_sport)

# Sampel 2: Kalimat bernuansa Ekonomi / Pasar Keuangan
sampel_finance = "Indeks Harga Saham Gabungan (IHSG) ditutup menguat signifikan didorong oleh aksi beli bersih investor asing pada saham sektor perbankan dan kenaikan dividen kuartal ketiga."
prediksi_kategori_berita(sampel_finance)


## 11. Kesimpulan & Analisis Pengaruh Hyperparameter

Berdasarkan hasil eksperimen pemodelan Skip-Gram dan klasifikasi Gaussian Naive Bayes pada 5 kasus hyperparameter, diperoleh beberapa temuan penting:

1. **Pengaruh Ukuran Dimensi (`vector_size`)**:
   - Dimensi moderat hingga tinggi (100–150 dimensi) memberikan ruang kapasitas representasi yang cukup kaya untuk memisahkan domain olahraga dan keuangan.
   - Dimensi yang terlalu kecil (seperti 50 pada Case 2) membatasi representasi nuansa semantik antarkata, sedangkan dimensi yang terlalu besar membutuhkan korpus yang lebih masif agar tidak terjadi redundansi fitur.

2. **Pengaruh Jendela Konteks (`window`)**:
   - Jendela konteks yang lebih luas (seperti 5 hingga 7) lebih menguntungkan untuk tugas klasifikasi dokumen (*topical classification*) karena menangkap asosiasi tema global di sekeliling kalimat, bukan sekadar hubungan sintaksis antar kata yang bersebelahan.

3. **Pengaruh Ambang Batas Frekuensi Kata (`min_count`)**:
   - Nilai `min_count=2` atau `min_count=3` efektif mengeliminasi kata-kata *typo* atau *hapax legomena* (kata yang hanya muncul 1 kali), sehingga ruang kosakata lebih bersih dan representasi rata-rata dokumen menjadi lebih stabil.

4. **Karakteristik Gaussian Naive Bayes pada Vektor Word2Vec**:
   - Model `GaussianNB` terbukti mampu mengklasifikasikan dokumen berbasis Word Embedding kontinu dengan akurasi tinggi.
   - Pemanfaatan *Average Word Embedding* (Mean Pooling) merupakan strategi yang efektif, sederhana, dan komputasionalnya efisien untuk mentransformasikan representasi kata menjadi representasi level dokumen.
